# 📘 Aula 07d: Visão Computacional Moderna – Redes Convolucionais (CNN) no PyTorch

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fboldt/aulasann/blob/main/aula07d_mnist_conv_torch.ipynb)

Chegamos ao ápice do nosso módulo comparativo de Visão Computacional!

Após implementarmos redes convolucionais no Keras (Aula 07b) e explorarmos a mecânica do PyTorch (Aula 07c), uniremos esses dois pilares: construir uma **Rede Neural Convolucional 2D Profunda em PyTorch** com `nn.Conv2d` e `nn.MaxPool2d` perfeitamente integrada aos **Pipelines do Scikit-Learn**.

---

### 🎯 Objetivo da Aula
1. Dominar a diferença fundamental de formato de tensores entre frameworks:
   * **Keras**: *Channels-Last* $\rightarrow (N, \text{altura}, \text{largura}, \text{canais})$
   * **PyTorch**: *Channels-First* $\rightarrow (N, \text{canais}, \text{altura}, \text{largura})$
2. Implementar o transformador `Shape2Torch` para converter tensores 2D em formato *Channels-First* com 1 canal de escala de cinza.
3. Compreender a dedução matemática das dimensões dos mapas de características: como uma entrada $28 \times 28$ se transforma após convoluções e poolings até resultar em $1.152$ ativações no Flatten.
4. Construir arquiteturas com `nn.Module` utilizando `nn.Conv2d`, `nn.MaxPool2d` e `nn.Flatten`.
5. Encapsular o treinamento em lotes (`DataLoader`) e a otimização com Adam dentro do estimador `TorchWrappedNN`, rodando diretamente em GPU ou CPU.

---

### 💡 A Intenção Pedagógica do Professor
* **Channels-First vs Channels-Last**: Chamar a atenção para o erro mais comum em visão computacional ao transitar entre Keras e PyTorch: a ordem do eixo de canais de cor.
* **Aritmética Manual de Convoluções**: Ensinar o estudante a calcular exatamente o tamanho do tensor intermediário para conectar a saída da última convolução à camada densa (`Linear(1152, num_classes)`).
* **Pipeline Universal**: Demonstrar que o Scikit-Learn pode orquestrar uma CNN de ponta em PyTorch com o mesmo padrão elegante de poucas linhas de código.

---

### 📚 Glossário da Aula
* **Channels-First**: Padrão de tensores do PyTorch onde a dimensão do canal vem antes da altura e largura: `(Batch, Channels, Height, Width)` ou `(N, C, H, W)`.
* **`nn.Conv2d`**: Camada convolucional bidimensional do PyTorch que aplica filtros espaciais deslizantes.
* **`nn.MaxPool2d`**: Camada de subamostragem que reduz as dimensões espaciais dividindo por um fator de escala (ex: corte de $2 \times 2$).
* **`nn.Flatten`**: Módulo que achata tensores multidimensionais a partir da dimensão 1, preservando o eixo do lote.

---

### 🧭 Trilha da Aula
1. [Carregamento dos Dados MNIST em Formato 2D](#1-carregamento-do-dataset-mnist)
2. [Seleção Automática do Dispositivo (CPU ou CUDA)](#2-gerenciamento-de-hardware)
3. [CNN PyTorch Inicial e Wrapper Básico](#3-cnn-pytorch-inicial-torchcnn2d)
4. [Channels-First: Transformador Customizado Shape2Torch](#4-o-padrão-channels-first-transformador-shape2torch)
5. [Pipeline Inicial com Shape2Torch e Convolução](#5-pipeline-com-normalização-e-cnn-pytorch)
6. [CNN Profunda de Alto Desempenho (32 -> 64 -> 128 Filtros)](#6-arquitetura-cnn-profunda-e-cálculo-de-dimensões)
7. [Pipeline Final de Alto Desempenho (Superando 99% de Acurácia)](#7-pipeline-final-completo-com-cnn-profunda)


## 1. Carregamento do Dataset MNIST

**O que estamos fazendo**: Carregar os dados oficiais do MNIST com matrizes 2D de $28 \times 28$ pixels.

**Por que fazer assim**: Para processar com convoluções no PyTorch, preservamos as duas dimensões espaciais das imagens.



In [ ]:
# Importa carregador do MNIST
from tensorflow.keras.datasets import mnist

# Carrega as partições de treino (60.000) e teste (10.000) em matrizes 2D (28x28)
(train_images, train_labels), (test_images, test_labels) = mnist.load_data()
print('Dimensão de treino:', train_images.shape)
print('Dimensão de teste: ', test_images.shape)


## 2. Gerenciamento de Hardware (CUDA / GPU)

**O que estamos fazendo**: Identificar se há GPU Nvidia disponível com suporte a CUDA ou executar via CPU.

**Por que fazer assim**: Operações de convolução 2D realizam milhões de multiplicações matriciais paralelas, beneficiando-se enormemente da aceleração em GPU.



In [ ]:
import torch

# Seleciona GPU se disponível; caso contrário, CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo ativo: {device}')


## 3. CNN PyTorch Inicial (TorchCNN2D) e Wrapper

**O que estamos fazendo**: Construir uma primeira arquitetura convolucional simples com `nn.Conv2d(1, 4, kernel_size=4)` e empacotá-la na classe `TorchWrappedNN`.

**Por que fazer assim**: Apresenta a estrutura mínima de convolução no PyTorch com Flatten e camadas lineares.

> [!NOTE]
> A camada `nn.Conv2d(in_channels=1, out_channels=4, kernel_size=4)` recebe 1 canal de entrada e gera 4 mapas de características de tamanho $(28 - 4 + 1) = 25 \times 25$.



In [ ]:
# Módulos essenciais do PyTorch e Scikit-Learn
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.metrics import accuracy_score

class TorchCNN2D(nn.Module):
  """
  Arquitetura CNN simples com 1 camada convolucional (4 filtros 4x4) e camadas densas.
  """
  def __init__(self, num_classes):
    super(TorchCNN2D, self).__init__()
    # Convolução 2D: entrada de 1 canal -> 4 canais de saída com kernel 4x4 (gera 4x25x25)
    self.conv1 = nn.Conv2d(in_channels=1, out_channels=4, kernel_size=4)
    self.flatten = nn.Flatten()
    # Camada densa: 4 * 25 * 25 = 2.500 entradas -> 512 neurônios
    self.fc1 = nn.Linear(in_features=4 * 25 * 25, out_features=512)
    # Camada de saída: 512 -> num_classes logits
    self.fc2 = nn.Linear(in_features=512, out_features=num_classes)

  def forward(self, x):
    x = self.conv1(x)
    x = F.relu(x)
    x = self.flatten(x)
    x = self.fc1(x)
    x = F.relu(x)
    x = self.fc2(x)
    return x

class TorchWrappedNN(BaseEstimator, ClassifierMixin):
  """
  Wrapper PyTorch compatível com Scikit-Learn para modelos convolucionais.
  """
  def __init__(self, epochs, batch_size, model_fabric=TorchCNN2D):
    self.epochs = epochs
    self.batch_size = batch_size
    self.model_fabric = model_fabric

  def fit(self, X, y):
    # Identifica rótulos de classe únicos
    self.labels, ids = torch.unique(torch.tensor(y), return_inverse=True)
    X_tensor = torch.from_numpy(X).float().to(device)
    y_tensor = torch.from_numpy(y).long().to(device)
    dataset = TensorDataset(X_tensor, y_tensor)
    loader = DataLoader(dataset, batch_size=self.batch_size, shuffle=True)

    # Instancia a CNN e prepara otimizador e perda
    self.model = self.model_fabric(len(self.labels)).to(device)
    self.optimizer = optim.Adam(self.model.parameters(), lr=0.001)
    self.criterion = nn.CrossEntropyLoss()

    # Loop de treino
    for epoch in range(self.epochs):
      for batch_idx, (images, labels) in enumerate(loader):
        self.optimizer.zero_grad()
        outputs = self.model(images)
        loss = self.criterion(outputs, labels)
        loss.backward()
        self.optimizer.step()
      print(f'Época [{epoch+1}/{self.epochs}] - Perda: {loss.item():.4f}')
    return self

  def predict(self, X):
    with torch.no_grad():
      X_tensor = torch.tensor(X, dtype=torch.float32).to(device)
      outputs = self.model(X_tensor)
      _, predicted = torch.max(outputs.data, 1)
      return self.labels[predicted.cpu().numpy()]


## 4. O Padrão Channels-First: Transformador Shape2Torch

**O que estamos fazendo**: Criar a classe `Shape2Torch(TransformerMixin)` para redimensionar as imagens de `(N, 28, 28)` para `(N, 1, 28, 28)`.

**Por que fazer assim**: O PyTorch exige tensores no formato `(Batch, Channels, Height, Width)`. Em escala de cinza, o canal 1 precisa estar explicitamente na dimensão de índice 1.

> [!IMPORTANT]
> Diferente do Keras onde o canal fica na última posição `(N, 28, 28, 1)`, no PyTorch o canal fica na segunda posição `(N, 1, 28, 28)`!



In [ ]:
from sklearn.base import TransformerMixin
from sklearn.pipeline import Pipeline

class Divide255(TransformerMixin):
  """
  Normaliza pixels dividindo por 255.0 para escala [0.0, 1.0].
  """
  def fit(self, X, y=None):
    return self

  def transform(self, X, y=None):
    return X / 255.0

class Shape2Torch(TransformerMixin):
  """
  Transformador Scikit-Learn que insere o canal no padrão Channels-First do PyTorch:
  (amostras, altura, largura) -> (amostras, 1, altura, largura)
  """
  def fit(self, X, y=None):
    return self

  def transform(self, X, y=None):
    # Insere o eixo do canal de cor na posição 1
    return X.reshape((X.shape[0], 1, X.shape[1], X.shape[2]))

# Constrói e testa o Pipeline inicial com a CNN simplificada
model = Pipeline([
  ('scaler', Divide255()),
  ('flatten', Shape2Torch()),
  ('nn', TorchWrappedNN(epochs=5, batch_size=128))
])
model.fit(train_images, train_labels)
y_pred = model.predict(test_images)
print(f'Acurácia Pipeline CNN Inicial: {accuracy_score(test_labels, y_pred) * 100:.2f}%')


## 5. Arquitetura CNN Profunda e Cálculo de Dimensões

**O que estamos fazendo**: Construir a classe `TorchCNN2D` profunda com 3 camadas convolucionais (`32`, `64` e `128` filtros) e 2 camadas de `MaxPool2d`.

**Por que fazer assim**: Essa é a arquitetura clássica completa, deduzindo a dimensão linear final ($1.152$ ativações).

> [!NOTE]
> Acompanhamento passo a passo das dimensões espaciais:
- Entrada: $(1, 28, 28)$
- Conv1 (32 filtros 3x3): $(32, 26, 26)$
- Pool1 (2x2): $(32, 13, 13)$
- Conv2 (64 filtros 3x3): $(64, 11, 11)$
- Pool2 (2x2): $(64, 5, 5)$
- Conv3 (128 filtros 3x3): $(128, 3, 3)$
- Flatten: $128 \times 3 \times 3 = 1.152$ elementos!
- Saída: `Linear(1152, num_classes)`.



In [ ]:
class TorchCNN2D(nn.Module):
  """
  CNN Profunda Completa em PyTorch para Visão Computacional de Alto Desempenho.
  Hierarquia: Conv(32) -> Pool -> Conv(64) -> Pool -> Conv(128) -> Flatten(1152) -> Linear(10).
  """
  def __init__(self, input_shape, num_classes):
    super(TorchCNN2D, self).__init__()
    # Bloco Convolucional 1: 1 canal -> 32 canais (kernel 3x3)
    self.conv1 = nn.Conv2d(in_channels=input_shape[0], out_channels=32, kernel_size=3)
    self.pool1 = nn.MaxPool2d(kernel_size=2)
    
    # Bloco Convolucional 2: 32 canais -> 64 canais (kernel 3x3)
    self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3)
    self.pool2 = nn.MaxPool2d(kernel_size=2)
    
    # Bloco Convolucional 3: 64 canais -> 128 canais (kernel 3x3)
    self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3)
    
    # Achatamento e classificação
    self.flatten = nn.Flatten()
    # 128 filtros * 3x3 resolução = 1152 neurônios de entrada
    self.fc1 = nn.Linear(in_features=1152, out_features=num_classes)

  def forward(self, x):
    x = self.conv1(x)
    x = F.relu(x)
    x = self.pool1(x)
    x = self.conv2(x)
    x = F.relu(x)
    x = self.pool2(x)
    x = self.conv3(x)
    x = F.relu(x)
    x = self.flatten(x)
    x = self.fc1(x)
    x = F.softmax(x, dim=1)
    return x

# Exibe a estrutura da rede instanciada
input_shape = (1, 28, 28)
num_classes = 10
model = TorchCNN2D(input_shape, num_classes)
print(model)


## 6. Pipeline Final Completo com CNN Profunda (99%+ de Acurácia)

**O que estamos fazendo**: Atualizar o `TorchWrappedNN` para inferir o `input_shape=X.shape[1:]` dinamicamente e executar o treinamento do Pipeline de alto desempenho.

**Por que fazer assim**: Alcança precisão superior a 99% demonstrando a paridade de excelência entre PyTorch e Keras.

> [!TIP]
> Observe como o código final no Scikit-Learn é idêntico ao do Keras: o padrão de engenharia desacoplada permitiu trocar o motor interno de Deep Learning mantendo a mesma interface de alto nível!



In [ ]:
class TorchWrappedNN(BaseEstimator, ClassifierMixin):
  """
  Wrapper PyTorch aprimorado com inferência dinâmica do formato dimensional de entrada.
  """
  def __init__(self, epochs, batch_size, model_fabric=TorchCNN2D):
    self.epochs = epochs
    self.batch_size = batch_size
    self.model_fabric = model_fabric

  def fit(self, X, y):
    self.labels, ids = torch.unique(torch.tensor(y), return_inverse=True)
    X_tensor = torch.from_numpy(X).float().to(device)
    y_tensor = torch.from_numpy(y).long().to(device)
    dataset = TensorDataset(X_tensor, y_tensor)
    loader = DataLoader(dataset, batch_size=self.batch_size, shuffle=True)

    # Instancia a CNN com input_shape (1, 28, 28) e número de classes
    self.model = self.model_fabric(input_shape=X.shape[1:], num_classes=len(self.labels)).to(device)
    self.optimizer = optim.Adam(self.model.parameters(), lr=0.001)
    self.criterion = nn.CrossEntropyLoss()

    for epoch in range(self.epochs):
      for batch_idx, (images, labels) in enumerate(loader):
        self.optimizer.zero_grad()
        outputs = self.model(images)
        loss = self.criterion(outputs, labels)
        loss.backward()
        self.optimizer.step()
      print(f'Época [{epoch+1}/{self.epochs}] - Perda: {loss.item():.4f}')
    return self

  def predict(self, X):
    with torch.no_grad():
      X_tensor = torch.tensor(X, dtype=torch.float32).to(device)
      outputs = self.model(X_tensor)
      _, predicted = torch.max(outputs.data, 1)
      return self.labels[predicted.cpu().numpy()]

# Executa o pipeline final ponta a ponta com a CNN Profunda em PyTorch
model = Pipeline([
  ('scaler', Divide255()),
  ('flatten', Shape2Torch()),
  ('nn', TorchWrappedNN(epochs=5, batch_size=128))
])
model.fit(train_images, train_labels)
y_pred = model.predict(test_images)
print(f'Acurácia Final da CNN Profunda (PyTorch): {accuracy_score(test_labels, y_pred) * 100:.2f}%')


---

## 🎓 Resumo da Aula & O Que Você Aprendeu

Nesta aula, consolidamos a implementação de Redes Convolucionais no framework **PyTorch**, completando o ciclo comparativo de Visão Computacional:

1. **Padrão Channels-First**: Compreendemos por que o PyTorch adota o formato $(N, C, H, W)$ e construímos o transformador `Shape2Torch` para alimentar a rede com $(N, 1, 28, 28)$.
2. **Aritmética dos Mapas Convolucionais**: Deduzimos a redução espacial provocada por convoluções sem padding e *Max Pooling*, demonstrando matematicamente como chegamos ao vetor de $1.152$ características.
3. **Modularidade com `nn.Module`**: Estruturamos uma CNN de três blocos convolucionais (`Conv32 -> Pool -> Conv64 -> Pool -> Conv128 -> Linear`).
4. **Superação dos 99% de Acurácia**: Confirmamos a alta precisão da arquitetura no conjunto de teste independente.
5. **Desacoplamento Profissional**: Provamos na prática que o padrão Scikit-Learn `Pipeline` e wrappers customizados permitem intercambiar Keras e PyTorch mantendo o mesmo fluxo de dados limpo e reprodutível.

---

## 👉 Próximo Passo
No **Lote 8 (Aulas 08a e 09a)**, avançaremos para o tratamento de **Imagens Reais Coloridas (Dataset Kaggle Dogs vs Cats)** com técnicas de Visão Computacional avançada e **Previsão de Séries Temporais (*Time Series Forecasting*)**!
